In [1]:
# libraries
import os
import json
import glob
import torch
import shutil
import random
import numpy as np
import nibabel as nib
from pathlib import Path
from multiprocessing import cpu_count
from joblib import Parallel, delayed

In [2]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
save_dir = "./Processed/"
shutil.rmtree(save_dir, ignore_errors=True)
os.makedirs(save_dir, exist_ok=True)

In [4]:
# random split the dataset into train and valid.
def random_split(inp_root_dir, split_ratio=0.2):

    split_dict = {}

    records = [r for r in os.listdir(inp_root_dir) if os.path.isdir(os.path.join(inp_root_dir, r))]
    # randomly shuffle the patient records for the split
    np.random.shuffle(records)

    # split indices
    split = int(len(records) * (1 - split_ratio))

    # split into train and valid
    train = records[:split]
    valid = records[split:]
    # store in dict
    split_dict['train'] = train
    split_dict['validation'] = valid

    print(f"Train : {len(train)}, Validation: {len(valid)}")

    # save in json file
    with open("split.json", "w") as file:
        json.dump(split_dict, file, indent=4)

    return split_dict

In [5]:
def process_case(args):

    case_id, inp_root_dir, save_dir, split_name = args

    print(f"Started {case_id}")
    case_path = os.path.join(inp_root_dir, case_id)
    nii_files = glob.glob(os.path.join(case_path, "*.nii.gz"))

    modalities = {}
    mask = None

    try:
        for file in nii_files:

            filename = os.path.basename(file)

            if "_seg.nii.gz" in filename:
                mask = nib.load(file).get_fdata()

            elif "_flair.nii.gz" in filename:
                modalities["flair"] = file

            elif "_t1ce.nii.gz" in filename:
                modalities["t1ce"] = file

            elif "_t1.nii.gz" in filename:
                modalities["t1"] = file

            elif "_t2.nii.gz" in filename:
                modalities["t2"] = file

        required = ["flair", "t1", "t1ce", "t2"]

        if not all(m in modalities for m in required):
            return f"{case_id}: missing modality"

        if mask is None:
            return f"{case_id}: missing mask"

        volumes = []

        # fixed BraTS order
        for mod in required:
            img = nib.load(modalities[mod]).get_fdata()
            volumes.append(img)

        images = np.stack(volumes, axis=0)

        # BUG FIX: validate mask shape matches image volume shape before saving
        if mask.shape != images.shape[1:]:
            return f"{case_id}: mask/image shape mismatch {mask.shape} vs {images.shape[1:]}"

        mask[mask == 4] = 3

        split_save_dir = os.path.join(save_dir, split_name)
        os.makedirs(split_save_dir, exist_ok=True)

        # BUG FIX: use forward-slash / POSIX-style paths consistently (avoids
        # Windows "\\" paths leaking into JSON metadata used cross-platform later)
        out_path = Path(split_save_dir) / f"{case_id}.npz"

        np.savez_compressed(out_path,
                             image=images.astype(np.float32),
                             mask=mask.astype(np.uint8))

        print(f"Ended {case_id}")
        return f"{case_id}: done"

    except Exception as e:
        # BUG FIX: catch per-case exceptions so one corrupt file doesn't kill
        # the whole parallel batch
        return f"{case_id}: FAILED ({e})"

In [6]:
def pre_process_images(inp_root_dir, split_dict, save_dir):

    os.makedirs(save_dir, exist_ok=True)

    # BUG FIX: actually use the computed worker count instead of hardcoding 4
    workers = min(cpu_count() - 1, 4)

    for split_name, cases in split_dict.items():
        print(f"\nProcessing {split_name} using {workers} workers")

        # BUG FIX: create the split directory once up front instead of
        # redundantly inside every worker call
        os.makedirs(os.path.join(save_dir, split_name), exist_ok=True)

        tasks = [(case_id, inp_root_dir, save_dir, split_name) for case_id in cases]

        results = Parallel(n_jobs=workers)(delayed(process_case)(task) for task in tasks)

        for r in results:
            print(r)

        print(f"{split_name} completed")

In [7]:
def create_slice_metadata(root_dir, split_dict, output_file, tumor_ratio=0.7, seed=42):

    random.seed(seed)
    root_dir = Path(root_dir)
    metadata = {}

    for split_name, records in split_dict.items():
        all_tumor_slices = []
        all_normal_slices = []

        # Collect all slices from all patients
        for patient_id in records:

            npz_file = root_dir / split_name / f"{patient_id}.npz"

            # BUG FIX: use context manager so the NpzFile handle is closed
            # after use instead of leaking across ~1000+ patients
            with np.load(npz_file) as data:
                mask = data["mask"]

                for slice_id in range(mask.shape[-1]):

                    mask_slice = mask[:, :, slice_id]
                    sample = {
                        # BUG FIX: store POSIX-style path string so metadata
                        # is portable across OSes
                        "file": npz_file.as_posix(),
                        "slice_id": slice_id
                    }

                    if np.any(mask_slice > 0):
                        sample["has_tumor"] = True
                        all_tumor_slices.append(sample)
                    else:
                        sample["has_tumor"] = False
                        all_normal_slices.append(sample)

        # Calculate required number of normal slices
        tumor_count = len(all_tumor_slices)
        normal_count = int(tumor_count * ((1 - tumor_ratio) / tumor_ratio))

        # Randomly sample normal slices
        sampled_normal_slices = random.sample(all_normal_slices, min(normal_count, len(all_normal_slices)))

        # Combine tumor + sampled normal slices
        final_samples = (all_tumor_slices + sampled_normal_slices)

        # Shuffle metadata
        random.shuffle(final_samples)
        metadata[split_name] = final_samples

        print(f"{split_name}:")
        print(f"Tumor slices  : {len(all_tumor_slices)}")
        print(f"Normal slices : {len(sampled_normal_slices)}")
        print(f"Total slices  : {len(final_samples)}")

    with open(output_file, "w") as f:
        json.dump(metadata, f, indent=4)

    print(f"Saved metadata → {output_file}")

In [8]:
split_dict = random_split(inp_root_dir="./Original/BraTS2021_Training_Data")

Train : 1000, Validation: 251


In [9]:
with open("split.json") as f:
    split_dict = json.load(f)

In [10]:
pre_process_images(inp_root_dir="./Original/BraTS2021_Training_Data",
                    split_dict=split_dict, save_dir=save_dir)



Processing train using 4 workers
BraTS2021_01137: done
BraTS2021_01659: done
BraTS2021_01483: done
BraTS2021_01317: done
BraTS2021_01523: done
BraTS2021_01633: done
BraTS2021_01098: done
BraTS2021_01128: done
BraTS2021_01117: done
BraTS2021_00281: done
BraTS2021_01577: done
BraTS2021_01422: done
BraTS2021_01059: done
BraTS2021_00804: done
BraTS2021_00077: done
BraTS2021_01040: done
BraTS2021_01456: done
BraTS2021_00607: done
BraTS2021_00746: done
BraTS2021_00219: done
BraTS2021_00297: done
BraTS2021_00588: done
BraTS2021_00601: done
BraTS2021_01532: done
BraTS2021_01393: done
BraTS2021_01602: done
BraTS2021_00110: done
BraTS2021_01332: done
BraTS2021_00382: done
BraTS2021_01204: done
BraTS2021_01387: done
BraTS2021_00112: done
BraTS2021_01531: done
BraTS2021_01212: done
BraTS2021_01351: done
BraTS2021_00734: done
BraTS2021_00419: done
BraTS2021_00063: done
BraTS2021_01627: done
BraTS2021_00097: done
BraTS2021_01036: done
BraTS2021_01230: done
BraTS2021_01400: done
BraTS2021_01549: don

In [11]:

create_slice_metadata(root_dir="Processed", split_dict=split_dict,
                       output_file="slice_metadata.json", tumor_ratio=0.7)

train:
Tumor slices  : 64462
Normal slices : 27626
Total slices  : 92088
validation:
Tumor slices  : 16975
Normal slices : 7275
Total slices  : 24250
Saved metadata → slice_metadata.json
